**1.ติดตั้งและยืนยันตัวตน GEE**

In [ ]:
import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
# Authenticate และ Initialize GEE
project_name = 'ee-lattytytyty' # User provided project name
try:
    ee.Initialize(project=project_name)
except ee.EEException as e:
    print(f"An Earth Engine error occurred: {e}")
    print("Attempting to authenticate and re-initialize...")
    ee.Authenticate() # This will prompt user for authentication
    try:
        ee.Initialize(project=project_name)
    except Exception as re_init_error:
        print(f"Error during re-initialization after authentication: {re_init_error}")
        raise
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    raise

An Earth Engine error occurred: Please authorize access to your Earth Engine account by running

earthengine authenticate

in your command line, or ee.Authenticate() in Python, and then retry.
Attempting to authenticate and re-initialize...


**2.กำหนดพื้นที่ศึกษา (AOI)**

เลือก “จังหวัดระยอง” เป็นพื้นที่วิเคราะห์

In [ ]:
# โหลดขอบเขตประเทศไทย
thailand = ee.FeatureCollection('FAO/GAUL/2015/level1') \
    .filter(ee.Filter.eq('ADM0_NAME', 'Thailand'))

# เลือกระยอง
rayong = thailand.filter(ee.Filter.eq('ADM1_NAME', 'Rayong'))

# ตั้ง AOI
aoi = rayong.geometry()

# แสดงแผนที่
Map = geemap.Map(center=[12.7, 101.4], zoom=9)
Map.addLayer(aoi, {'color': 'blue'}, 'Rayong')
Map

Map(center=[12.7, 101.4], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI…

In [ ]:
# ใช้ FAO GAUL Level-2 (district boundaries) หรือ geometry ตรง ๆ

import ee

# Rayong Province boundary จาก FAO GAUL
rayong = (ee.FeatureCollection("FAO/GAUL/2015/level1")
            .filter(ee.Filter.eq('ADM1_NAME', 'Rayong')))

# แปลงเป็น geometry สำหรับ clip/mask
roi = rayong.geometry()

# ตรวจสอบ
print("Study Area: Rayong Province, Thailand")
print("Bounds:", roi.bounds().getInfo())

Study Area: Rayong Province, Thailand
Bounds: {'geodesic': False, 'type': 'Polygon', 'coordinates': [[[100.98572530237223, 12.52136088971078], [101.83173273581755, 12.52136088971078], [101.83173273581755, 13.159476071762525], [100.98572530237223, 13.159476071762525], [100.98572530237223, 12.52136088971078]]]}


**3.โหลดชุดข้อมูลทั้งหมด**

In [ ]:
# โหลดข้อมูลทุก layer
# แต่ละ dataset มีที่มาและเหตุผลชัดเจน

# 1. SRTM Digital Elevation Model (30m) — NASA
dem = ee.Image("USGS/SRTMGL1_003").clip(roi)

# 2. Sentinel-2 Surface Reflectance (2023) — ESA
s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterDate('2023-01-01', '2023-12-31')
        .filterBounds(roi)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        .median()
        .clip(roi))

# 3. HydroSHEDS Stream Network — WWF
rivers = ee.FeatureCollection("WWF/HydroSHEDS/v1/FreeFlowingRivers").filterBounds(roi)

# 4. WorldPop Population Density 2020 (100m) — University of Southampton
population = (ee.ImageCollection("WorldPop/GP/100m/pop")
                .filter(ee.Filter.eq('country', 'THA'))
                .filter(ee.Filter.eq('year', 2020))
                .first()
                .clip(roi))

# 5. OpenStreetMap Industrial Areas — ผ่าน GEE Community Dataset
#    ใช้ Dynamic World แทน (real-time land cover) เพราะ OSM ใน GEE มีจำกัด
land_cover = (ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
                .filterDate('2023-01-01', '2023-12-31')
                .filterBounds(roi)
                .median()
                .clip(roi))

print("✅ All datasets loaded")
print("Bands available in Sentinel-2:", s2.bandNames().getInfo())
print("Bands available in Dynamic World:", land_cover.bandNames().getInfo())

✅ All datasets loaded
Bands available in Sentinel-2: ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE']
Bands available in Dynamic World: ['water', 'trees', 'grass', 'flooded_vegetation', 'crops', 'shrub_and_scrub', 'built', 'bare', 'snow_and_ice', 'label']


**4. คำนวณค่า NDVI (ปัจจัยที่ 1)**

In [ ]:
# Cell 4: คำนวณ NDVI จาก Sentinel-2
# NDVI = (NIR - Red) / (NIR + Red) = (B8 - B4) / (B8 + B4)
# เหตุผล: พื้นที่อุตสาหกรรมมี NDVI ต่ำ (< 0.2) บ่งชี้พื้นที่ impervious
# อ้างอิง: Tucker (1979), Remote Sensing of Environment

ndvi = s2.normalizedDifference(['B8', 'B4']).rename('NDVI')

# Normalize: Invert NDVI → พื้นที่ NDVI ต่ำ = risk สูง
# formula: ndvi_risk = 1 - ((NDVI - min) / (max - min))
# พื้นที่ระยองมี NDVI ประมาณ -0.1 ถึง 0.8
ndvi_norm = ndvi.unitScale(-0.1, 0.8)           # scale 0-1
ndvi_risk = ee.Image(1).subtract(ndvi_norm)     # invert: low NDVI = high risk
ndvi_risk = ndvi_risk.rename('ndvi_risk').clamp(0, 1)

print("✅ NDVI risk layer computed")
print("Logic: low vegetation (industrial/bare land) → higher pollution risk")

✅ NDVI risk layer computed
Logic: low vegetation (industrial/bare land) → higher pollution risk


**5.ระยะทางถึงแม่น้ำ (ปัจจัย 2)**

In [ ]:
# ระยะห่างจากลำน้ำ — ยิ่งใกล้แม่น้ำ = เสี่ยงรับมลพิษสูง
# เหตุผล: มลพิษในน้ำแพร่กระจายตามลำน้ำ พื้นที่ริมคลองได้รับผลกระทบก่อน
# อ้างอิง: Allan & Castillo (2007) Stream Ecology

# Rasterize river network เป็น binary image ก่อน
river_image = rivers.reduceToImage(
    properties=['ORD_STRA'],
    reducer=ee.Reducer.first()
).unmask(0).gt(0).rename('rivers')

# Euclidean distance (เมตร) จากแต่ละ pixel ถึงลำน้ำ
river_distance = river_image.fastDistanceTransform().sqrt().multiply(
    ee.Image.pixelArea().sqrt()  # แปลงหน่วยเป็นเมตร
).rename('river_distance_m')

# Normalize: ใกล้แม่น้ำ = risk สูง → invert distance
# กำหนด max distance = 10,000 เมตร (10 km)
MAX_DIST = 10000
river_dist_norm = river_distance.unitScale(0, MAX_DIST).clamp(0, 1)
river_risk = ee.Image(1).subtract(river_dist_norm)
river_risk = river_risk.rename('river_risk')

print("✅ River proximity risk layer computed")
print("Logic: closer to rivers → higher risk of contamination spread")

✅ River proximity risk layer computed
Logic: closer to rivers → higher risk of contamination spread


**6.การสะสมการไหลจาก DEM (ปัจจัยที่ 3)**

In [ ]:
# Flow Accumulation — บ่งชี้ทิศทางการไหลของน้ำและจุดสะสมมลพิษ
# เหตุผล: Downstream areas สะสมมลพิษมากกว่า upstream
# อ้างอิง: O'Callaghan & Mark (1984) — D8 flow algorithm

# คำนวณ slope จาก DEM
slope = ee.Terrain.slope(dem).rename('slope')

# ใช้ HydroSHEDS Basins + Upstream drainage area เป็น proxy
# (GEE ไม่มี flow accumulation โดยตรง ใช้ Drainage Basin แทน)
hydro_basins = ee.FeatureCollection("WWF/HydroSHEDS/v1/Basins/hybas_7").filterBounds(roi)

# Slope ต่ำ = พื้นที่ราบลุ่ม = น้ำขัง = risk สูง
# Normalize slope แบบ inverse: slope ต่ำ = risk สูง
slope_norm = slope.unitScale(0, 30).clamp(0, 1)    # slope 0-30 องศาในระยอง
slope_risk = ee.Image(1).subtract(slope_norm)       # invert
slope_risk = slope_risk.rename('slope_risk')

print("✅ Topographic risk layer computed")
print("Logic: low slope (flat/downstream areas) → water & pollutants accumulate")

✅ Topographic risk layer computed
Logic: low slope (flat/downstream areas) → water & pollutants accumulate


**7.ความหนาแน่นของประชากร (ปัจจัยที่ 4)**

In [ ]:
# ความหนาแน่นประชากร — ยิ่งหนาแน่น = ผลกระทบสาธารณสุขสูง
# เหตุผล: มลพิษน้ำกระทบประชากรที่อยู่ใกล้แหล่งน้ำโดยตรง
# อ้างอิง: WorldPop (Stevens et al., 2015)

# Log-transform population density ก่อน normalize
# เพราะการกระจายตัวของประชากรแบบ skewed
pop_log = population.add(1).log().rename('pop_log')     # log(1 + x) หลีกเลี่ยง log(0)

# Normalize 0-1 (log scale)
pop_min = 0
pop_max = 10    # log(1+22000) ≈ 10 (urban core max)
pop_norm = pop_log.unitScale(pop_min, pop_max).clamp(0, 1)
pop_risk = pop_norm.rename('pop_risk')

print("✅ Population density risk layer computed")
print("Logic: high population density near waterways → greater health impact")

✅ Population density risk layer computed
Logic: high population density near waterways → greater health impact


**8.พื้นที่ใช้ประโยชน์ที่ดินเพื่ออุตสาหกรรม (ปัจจัยที่ 5)**

In [ ]:
# พื้นที่อุตสาหกรรม จาก Dynamic World
# Class 6 = Built area (industrial + urban)
# เหตุผล: แหล่งกำเนิดมลพิษหลักมาจาก built-up industrial zones
# อ้างอิง: Brown et al. (2022) Dynamic World, Nature Communications

# Dynamic World label band: 0-8 classes
# class 6 = 'built' (urban/industrial)
dw_label = land_cover.select('label')
industrial_mask = dw_label.eq(6).rename('industrial')

# ระยะห่างจากพื้นที่ built (เมตร) — ยิ่งใกล้ = เสี่ยงสูง
industrial_distance = industrial_mask.fastDistanceTransform().sqrt().multiply(
    ee.Image.pixelArea().sqrt()
).rename('industrial_distance_m')

MAX_IND_DIST = 5000   # 5 km radius of influence
ind_dist_norm = industrial_distance.unitScale(0, MAX_IND_DIST).clamp(0, 1)
industrial_risk = ee.Image(1).subtract(ind_dist_norm)
industrial_risk = industrial_risk.rename('industrial_risk')

print("✅ Industrial proximity risk layer computed")
print("Logic: proximity to built/industrial areas → higher pollution source probability")

✅ Industrial proximity risk layer computed
Logic: proximity to built/industrial areas → higher pollution source probability


**9. การรวมแบบถ่วงน้ำหนัก (แบบจำลอง WLC)**

In [ ]:
# Weighted Linear Combination (WLC)
# สูตร: Risk = w1*NDVI_risk + w2*River_risk + w3*Slope_risk + w4*Pop_risk + w5*Ind_risk
#
# น้ำหนักมาจาก: Expert Knowledge + AHP (Analytic Hierarchy Process)
# อ้างอิง: Saaty (1980) AHP + Malczewski (1999) GIS-based Multicriteria Analysis
#
# เหตุผลน้ำหนัก:
# - Industrial proximity (0.35): แหล่งกำเนิดมลพิษโดยตรง — น้ำหนักสูงสุด
# - River proximity (0.25): เส้นทางแพร่กระจายหลัก
# - Population (0.20): ตัวชี้วัดผลกระทบสาธารณสุข
# - Slope/Topography (0.12): กำหนดทิศทางการไหล
# - NDVI (0.08): ตัวชี้วัดสภาพพื้นที่สนับสนุน
# ผลรวม = 1.00 ✓

w_industrial = 0.35
w_river      = 0.25
w_population = 0.20
w_slope      = 0.12
w_ndvi       = 0.08

assert abs(w_industrial + w_river + w_population + w_slope + w_ndvi - 1.0) < 1e-9, \
    "Weights must sum to 1.0"

# WLC computation
risk_index = (industrial_risk.multiply(w_industrial)
              .add(river_risk.multiply(w_river))
              .add(pop_risk.multiply(w_population))
              .add(slope_risk.multiply(w_slope))
              .add(ndvi_risk.multiply(w_ndvi))
              ).rename('WasteWater_Risk_Index')

print("✅ WLC model computed")
print(f"Weights: Industrial={w_industrial}, River={w_river}, "
      f"Pop={w_population}, Slope={w_slope}, NDVI={w_ndvi}")
print("Sum of weights:", w_industrial + w_river + w_population + w_slope + w_ndvi)

✅ WLC model computed
Weights: Industrial=0.35, River=0.25, Pop=0.2, Slope=0.12, NDVI=0.08
Sum of weights: 1.0


**10. จำแนกความเสี่ยงและแสดงภาพแผนที่**

In [ ]:
# แบ่งระดับความเสี่ยง และแสดงแผนที่
# Threshold มาจาก: Equal Interval (เพราะ normalized 0-1) + วรรณกรรม
# อ้างอิง: Ayalew & Yamagishi (2005) — risk classification in GIS

import geemap

# แบ่ง 4 ระดับ
risk_classified = (risk_index
    .where(risk_index.lte(0.25), 1)   # ต่ำ (Low)
    .where(risk_index.gt(0.25).And(risk_index.lte(0.50)), 2)   # ปานกลาง
    .where(risk_index.gt(0.50).And(risk_index.lte(0.75)), 3)   # สูง
    .where(risk_index.gt(0.75), 4)    # สูงมาก
    .rename('Risk_Class'))

# Visualization parameters
risk_vis = {
    'min': 0, 'max': 1,
    'palette': ['#1a9641', '#a6d96a', '#fdae61', '#d7191c']  # เขียว → เหลือง → ส้ม → แดง
}

classified_vis = {
    'min': 1, 'max': 4,
    'palette': ['#1a9641', '#a6d96a', '#fdae61', '#d7191c']
}

# สร้าง interactive map
Map = geemap.Map()
Map.centerObject(roi, zoom=10)
Map.addLayer(risk_index, risk_vis, 'Wastewater Risk Index (Continuous)')
Map.addLayer(risk_classified, classified_vis, 'Risk Classification (4 Classes)')
Map.addLayer(rayong, {}, 'Rayong Boundary', opacity=0.5)

# Legend
Map.add_colorbar(risk_vis, label='Wastewater Pollution Risk (0=Low, 1=High)')

Map

Map(center=[12.854474761046609, 101.42792601809239], controls=(WidgetControl(options=['position', 'transparent…

**11. การวิเคราะห์ความไว (±20%)**

In [ ]:
# Sensitivity Analysis
# ทดสอบ: เปลี่ยน w_industrial ±20% แล้วดูว่าแผนที่เปลี่ยนมากแค่ไหน
# อ้างอิง: Saltelli et al. (2008) Global Sensitivity Analysis

import numpy as np

def compute_risk(w_ind, w_riv, w_pop, w_slo, w_ndv):
    """คำนวณ Risk Index จาก weights ที่กำหนด"""
    # Normalize weights ให้ผลรวม = 1 เสมอ
    total = w_ind + w_riv + w_pop + w_slo + w_ndv
    return (industrial_risk.multiply(w_ind/total)
            .add(river_risk.multiply(w_riv/total))
            .add(pop_risk.multiply(w_pop/total))
            .add(slope_risk.multiply(w_slo/total))
            .add(ndvi_risk.multiply(w_ndv/total))
            ).rename('Risk')

# Base case
risk_base = compute_risk(0.35, 0.25, 0.20, 0.12, 0.08)

# Scenario +20% industrial weight
risk_high_ind = compute_risk(0.35*1.2, 0.25, 0.20, 0.12, 0.08)

# Scenario -20% industrial weight
risk_low_ind  = compute_risk(0.35*0.8, 0.25, 0.20, 0.12, 0.08)

# Difference maps
diff_high = risk_high_ind.subtract(risk_base).abs().rename('diff_high')
diff_low  = risk_low_ind.subtract(risk_base).abs().rename('diff_low')

# วิเคราะห์ % พื้นที่ที่เปลี่ยนระดับความเสี่ยง
diff_threshold = 0.1   # เปลี่ยน > 0.1 ถือว่า sensitive

sensitive_area = diff_high.gt(diff_threshold)
robust_area    = diff_high.lte(diff_threshold)

# คำนวณ area statistics
def get_area_km2(image, roi):
    area = image.multiply(ee.Image.pixelArea()).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=roi,
        scale=100,
        maxPixels=1e10
    )
    return area

sensitive_km2 = get_area_km2(sensitive_area, roi)
print("Area sensitive to ±20% change in industrial weight:")
print(sensitive_km2.getInfo())

# Visualize sensitivity
sens_vis = {'min': 0, 'max': 0.2, 'palette': ['white', 'orange', 'red']}
Map2 = geemap.Map()
Map2.centerObject(roi, zoom=10)
Map2.addLayer(diff_high, sens_vis, 'Sensitivity to +20% Industrial Weight')
Map2.addLayer(risk_base, risk_vis, 'Base Risk (reference)')
Map2

Area sensitive to ±20% change in industrial weight:
{'diff_high': 0}


Map(center=[12.854474761046657, 101.4279260180924], controls=(WidgetControl(options=['position', 'transparent_…

**12.การตรวจสอบความถูกต้องเทียบกับน้ำผิวดินของ JRC**

In [ ]:
# Validation — เปรียบเทียบกับ JRC Global Surface Water
# เหตุผล: พื้นที่น้ำท่วมถาวร/ฤดูกาล = ตัวแทน waterway contamination zones
# อ้างอิง: Pekel et al. (2016) Nature — JRC Global Surface Water

# โหลด JRC Global Surface Water
jrc = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").clip(roi)

# occurrence > 50% = พื้นที่มีน้ำบ่อย (น้ำถาวรหรือน้ำตามฤดู)
water_mask = jrc.select('occurrence').gt(50).rename('water_presence')

# ความเสี่ยงสูง (class 3-4) ควรทับซ้อนกับพื้นที่ริมน้ำ
high_risk_mask = risk_classified.gte(3)

# Confusion-style overlap analysis
# True Positive: high risk AND water presence
TP = high_risk_mask.And(water_mask)
FP = high_risk_mask.And(water_mask.Not())
FN = high_risk_mask.Not().And(water_mask)
TN = high_risk_mask.Not().And(water_mask.Not())

def count_pixels(image, roi, scale=100):
    count = image.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=roi,
        scale=scale,
        maxPixels=1e10
    )
    return count.getInfo()

tp_count = count_pixels(TP, roi)
fp_count = count_pixels(FP, roi)
fn_count = count_pixels(FN, roi)
tn_count = count_pixels(TN, roi)

print("=== Validation Results ===")
print(f"True Positive  (High Risk & Water Present): {tp_count}")
print(f"False Positive (High Risk & No Water):      {fp_count}")
print(f"False Negative (Low Risk  & Water Present): {fn_count}")
print(f"True Negative  (Low Risk  & No Water):      {tn_count}")

# คำนวณ Precision & Recall (เชิง spatial overlap)
# Precision = TP / (TP + FP)
# Recall    = TP / (TP + FN)
# หมายเหตุ: นี้เป็น proxy validation เท่านั้น ไม่ใช่ ground truth จริง

# Visualize
Map3 = geemap.Map()
Map3.centerObject(roi, zoom=10)
Map3.addLayer(risk_classified, classified_vis, 'Risk Classification')
Map3.addLayer(water_mask, {'min':0,'max':1,'palette':['white','blue']}, 'JRC Water Presence')
Map3.addLayer(TP, {'min':0,'max':1,'palette':['white','darkred']}, 'True Positive (overlap)')
Map3

=== Validation Results ===
True Positive  (High Risk & Water Present): {'Risk_Class': 1588.7176470588272}
False Positive (High Risk & No Water):      {'Risk_Class': 752.196078431348}
False Negative (Low Risk  & Water Present): {'Risk_Class': 2052.7882352940983}
True Negative  (Low Risk  & No Water):      {'Risk_Class': 544.6549019607711}


Map(center=[12.854474761046609, 101.42792601809239], controls=(WidgetControl(options=['position', 'transparent…

**13.Export ผลลัพธ์ไปยัง Google Drive**

In [ ]:
# Cell 13: Export ผลลัพธ์ไปยัง Google Drive
# สำหรับสร้าง figures/ ใน GitHub submission

# Export 1: Risk Index (continuous)
task1 = ee.batch.Export.image.toDrive(
    image=risk_index,
    description='Rayong_WastewaterRisk_Continuous',
    folder='GEE338_Lab4',
    fileNamePrefix='rayong_risk_index',
    region=roi,
    scale=100,
    crs='EPSG:32648',   # UTM Zone 48N — เหมาะสำหรับประเทศไทย
    maxPixels=1e10
)
task1.start()

# Export 2: Risk Classification (4 classes)
task2 = ee.batch.Export.image.toDrive(
    image=risk_classified,
    description='Rayong_RiskClassified_4Class',
    folder='GEE338_Lab4',
    fileNamePrefix='rayong_risk_classified',
    region=roi,
    scale=100,
    crs='EPSG:32648',
    maxPixels=1e10
)
task2.start()

# Export 3: Sensitivity diff map
task3 = ee.batch.Export.image.toDrive(
    image=diff_high,
    description='Rayong_SensitivityAnalysis',
    folder='GEE338_Lab4',
    fileNamePrefix='rayong_sensitivity',
    region=roi,
    scale=100,
    crs='EPSG:32648',
    maxPixels=1e10
)
task3.start()

print("✅ Export tasks submitted to Google Drive → folder: GEE338_Lab4")
print("ตรวจสอบ progress ที่: https://code.earthengine.google.com/tasks")

✅ Export tasks submitted to Google Drive → folder: GEE338_Lab4
ตรวจสอบ progress ที่: https://code.earthengine.google.com/tasks
